# Project 2: Predictive Modeling Using Machine Learning
**Thiranex Data Science Internship Portfolio**  
*Author:* Data Science Intern  
*Dataset:* Breast Cancer Wisconsin (Diagnostic) Dataset  

---
## 1. Project Overview & Clinical Context
Supervised machine learning in healthcare requires extreme vigilance regarding metric selection, pipeline architecture, and data leakage. In oncology screening, **Recall (Sensitivity)** on malignant cases is of paramount importance: failing to detect a malignant tumor (False Negative) has catastrophic clinical consequences, whereas a False Positive simply prompts a routine follow-up biopsy.

> **Clinical Disclaimer:** This notebook and associated models represent an educational machine learning portfolio project. It is **not** certified as a clinical diagnostic tool.

### Key Objectives:
1. Implement stratified train-test splitting preserving exact positive class proportions.
2. Construct Scikit-Learn `Pipeline` architectures that fit scalers **only** on training data to strictly prevent data leakage.
3. Train and compare 4 distinct algorithmic paradigms: Logistic Regression, Decision Tree, Random Forest, and Support Vector Machine.
4. Evaluate models via Accuracy, Precision, Recall, F1-Score, Confusion Matrices, and ROC-AUC.
5. Serialize the best-performing pipeline using `joblib`.


In [1]:
# 1. Environment & Library Imports
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('All libraries imported successfully!')

In [2]:
# 2. Ingest Dataset & Persist Raw CSV
cancer = load_breast_cancer()
df = pd.DataFrame(data=cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

raw_path = '../data/breast_cancer_raw.csv'
os.makedirs(os.path.dirname(raw_path), exist_ok=True)
df.to_csv(raw_path, index=False)

print(f'Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')
print('Class Distribution:')
print(df['diagnosis'].value_counts())
df.head()

In [3]:
# 3. Stratified Train-Test Splitting (Preventing Leakage)
feature_cols = cancer.feature_names.tolist()
X = df[feature_cols]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training Set: {X_train.shape[0]} samples | Testing Set: {X_test.shape[0]} samples')
print(f'Train Malignant Proportion: {(y_train == 0).mean() * 100:.2f}%')
print(f'Test Malignant Proportion:  {(y_test == 0).mean() * 100:.2f}%')

In [4]:
# 4. Training 4 Models with StandardScaler Pipelines
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_split=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    'Support Vector Machine': SVC(C=1.0, kernel='rbf', probability=True, random_state=42)
}

results = {}
fitted_pipes = {}

for name, clf in models.items():
    # StandardScaler fitted ONLY on training data inside pipeline
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 0]  # Probability of Malignant (class 0)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label=0)
    rec = recall_score(y_test, y_pred, pos_label=0)
    f1 = f1_score(y_test, y_pred, pos_label=0)
    auc = roc_auc_score(y_test == 0, y_prob)
    
    results[name] = {
        'Accuracy': acc,
        'Precision (Mal)': prec,
        'Recall (Sensitivity)': rec,
        'F1-Score': f1,
        'ROC-AUC': auc,
        'cm': confusion_matrix(y_test, y_pred),
        'y_prob': y_prob
    }
    fitted_pipes[name] = pipe

df_results = pd.DataFrame([
    {'Algorithm': k, 'Accuracy': v['Accuracy'], 'Precision (Mal)': v['Precision (Mal)'],
     'Recall (Sensitivity)': v['Recall (Sensitivity)'], 'F1-Score': v['F1-Score'], 'ROC-AUC': v['ROC-AUC']}
    for k, v in results.items()
]).sort_values(by='F1-Score', ascending=False)

print('=== MODEL BENCHMARK RESULTS ===')
df_results

In [5]:
# 5. Confusion Matrix Visualization
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    sns.heatmap(
        res['cm'], annot=True, fmt='d', cmap='Blues', cbar=False,
        xticklabels=['Malignant (0)', 'Benign (1)'],
        yticklabels=['Malignant (0)', 'Benign (1)'],
        ax=axes[i]
    )
    axes[i].set_title(f"{name}\nAccuracy: {res['Accuracy']:.3f} | Recall: {res['Recall (Sensitivity)']:.3f}", weight='bold')
    axes[i].set_xlabel('Predicted Diagnosis')
    axes[i].set_ylabel('True Diagnosis')

plt.tight_layout()
plt.show()

In [6]:
# 6. Combined ROC Curves
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#2563EB', '#EF4444', '#10B981', '#7C3AED']

for i, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test == 0, res['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC = {res['ROC-AUC']:.4f})", color=colors[i], linewidth=2.2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Chance (AUC = 0.50)')
ax.set_title('Receiver Operating Characteristic (ROC) Curves', fontsize=13, weight='bold')
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity / Recall)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [7]:
# 7. Feature Importance Analysis (Random Forest)
rf = fitted_pipes['Random Forest'].named_steps['classifier']
feat_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='Blues_r', ax=ax)
ax.set_title('Top 10 Most Predictive Cellular Features', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

In [8]:
# 8. Model Serialization
model_save_path = '../models/best_model.joblib'
os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
joblib.dump(fitted_pipes['Logistic Regression'], model_save_path)
print(f'Model pipeline saved to: {model_save_path}')

## 9. Key Findings & Discussion
1. **Linear Separability in Normalized Space:** Standardized cellular features produced linear separability, enabling Logistic Regression and RBF-SVM to achieve **98.25% test accuracy** and **97.62% sensitivity**.
2. **Criticality of Recall:** Across 42 malignant test cases, the best models missed only 1 case (False Negative rate = 2.38%), which is the critical safety benchmark in medical screening.
3. **Zero Data Leakage:** Encapsulating standard scaling inside the Scikit-learn Pipeline ensured test set statistics remained completely unobserved during training.
